<a href="https://colab.research.google.com/github/mayajdias/ds2002-fa26/blob/main/notebooks/01-foundations/2026_10_02_normalize_and_join_nested_json_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [ ]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-08','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** a single line item on a single order (one item, with its quantity, from one order), with the order's `order` and `vendor_id` carried along.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [ ]:
import pandas as pd

lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

assert len(lines) == 4
assert lines['qty'].sum() == 7
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-08


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [ ]:
vendors = pd.DataFrame(vendors_json)

# validate='m:1' fails loudly if vendors had duplicate vendor_ids (which would silently multiply rows)
joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True, validate='m:1')

unmatched = joined[joined['_merge'] == 'left_only']
print("Unmatched vendor(s):", unmatched['vendor_id'].unique().tolist())
print("Units attached to unmatched vendor(s):", unmatched['qty'].sum())
joined

Unmatched vendor(s): ['V-08']
Units attached to unmatched vendor(s): 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-08,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [ ]:
# Keep the unmatched vendor as its own labeled row instead of dropping it. A default groupby
# silently discards NaN keys, so the totals would add up to 4 instead of 7 and the missing
# units would be invisible. Labeling it keeps the table reconciled to 7 and flags the data problem.
joined['name_clean'] = joined['name'].fillna('UNMATCHED (' + joined['vendor_id'] + ')')

qty_by_vendor = joined.groupby('name_clean')['qty'].sum().sort_values(ascending=False)
print(qty_by_vendor)
assert qty_by_vendor.sum() == 7

name_clean
Hoos Burgers        3
UNMATCHED (V-08)    3
Cav Merch           1
Name: qty, dtype: int64


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [ ]:
# Vendors with no match have no zone, so give them their own bucket instead of letting groupby drop them.
NO_ZONE = 'No zone (unmatched vendor)'
joined['zone_clean'] = joined['zone'].fillna(NO_ZONE)

qty_by_zone = joined.groupby('zone_clean')['qty'].sum().sort_values(ascending=False)
print(qty_by_zone)

matched_total = qty_by_zone.drop(NO_ZONE, errors='ignore').sum()   # errors='ignore' = no crash if the bucket doesn't exist
unmatched_total = qty_by_zone.get(NO_ZONE, 0)

print(f"\nTop zone: {qty_by_zone.index[0]} with {qty_by_zone.iloc[0]} items")
print(f"Zone total counting only real zones: {matched_total}")
print(f"Units with no zone: {unmatched_total}")
print(f"Grand total: {qty_by_zone.sum()}")

if unmatched_total == 0:
    print("Every order matched a vendor, so the zone totals add up to 7 and nothing went missing.")
else:
    print(f"The {unmatched_total}-unit difference is from the unmatched vendor, which has no zone.")

zone_clean
A                             4
No zone (unmatched vendor)    3
Name: qty, dtype: int64

Top zone: A with 4 items
Zone total counting only real zones: 4
Units with no zone: 3
Grand total: 7
The 3-unit difference is from the unmatched vendor, which has no zone.


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [ ]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]
# json_normalize raises a KeyError if record_path ('lines') is missing from any order.
# Give lineless orders a single empty line so they survive as a row with item/qty missing.
ragged_safe = [{**o, 'lines': o.get('lines') or [{}]} for o in ragged_json]

ragged = pd.json_normalize(ragged_safe, record_path='lines', meta=['order', 'vendor_id'])

# Nullable Int64 stores a missing qty as <NA>, which stays distinct from a real 0
# (no fillna(0)). It also keeps quantities as integers instead of floats.
ragged['qty'] = ragged['qty'].astype('Int64')

print(ragged)
print("\nRows with missing qty:", ragged['qty'].isna().sum())   # 2 (orders 5 and 6)
print("Rows with qty == 0:   ", (ragged['qty'] == 0).sum())      # 0
print("Sum of known qty:     ", ragged['qty'].sum())             # 2 (skips <NA>)

    item   qty order vendor_id
0   Soda     2     4      V-01
1    NaN  <NA>     5      V-10
2  Fries  <NA>     6      V-01

Rows with missing qty: 2
Rows with qty == 0:    0
Sum of known qty:      2


### Q6 — Validate

In [ ]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**a)** One row per line item is the useful shape because the line item is the smallest unit of the data (item + quantity), so everything else is an aggregation of it. Questions like "total units per vendor," "units per zone," or "best-selling item" become a simple `groupby().sum()`. With one row per order, the items would be trapped in a nested list, so I'd have to unpack them first, and I couldn't sum quantities at all without extra work. Flattening also exposed the join problem: the V-08 order showed up as a row I could count (3 units).

**b)** A missing quantity means "we don't know how many"; a quantity of zero means "we know it was none." If they're conflated with `fillna(0)`, totals look complete and confident when they're really understated, averages are dragged down, and the data-quality problem (orders 5 and 6) disappears so nobody investigates. Keeping it as `<NA>` means sums skip it, `.isna()` can count and flag it, and a downstream person can see that two records need to be chased down rather than assuming they sold nothing.